In [ ]:
import os

from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI
import requests

In [2]:
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

anthropic_api_url = "https://api.anthropic.com/v1/"

openai = OpenAI(api_key=openai_api_key)
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_api_url)

In [3]:
def fetch_website_contents(url: str) -> str | None:
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
    }
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        return None
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return title + "\n\n" + text

In [4]:
def fetch_wikipedia_page(subject:str) -> str | None:
    url = f"https://en.wikipedia.org/wiki/{subject.lower().strip().replace(' ', '_').replace("-", "_")}"
    return fetch_website_contents(url)

In [18]:
city = "Paris"

In [ ]:
page = fetch_wikipedia_page(city)

In [61]:
def summarize_text(text: str, model: str = "gpt-5.6-luna") -> str | None:
    system_prompt = """You are a helpful assistant that summarizes text in a very concise (less or equal to 2000 characters), structured and compelling way,
    ignoring text that might be navigation related. 
    Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown."""
    user_prompt = f"Summarize the following text in a very concise (less or equal to 2000 characters), structured and compelling way:\n\n{text}"
    response = openai.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
    )
    result = response.choices[0].message.content
    return result[:4096] if result else None

In [62]:
if page:
    summarized_page = summarize_text(page)

In [41]:
def find_out_the_main_spoken_language_in_city(city: str, model: str = "gpt-5.6-luna") -> str | None:
    system_prompt = """You are a helpful assistant that finds out the main language spoken in a city.
    Respond with the name of the language only, without any additional text."""
    user_prompt = f"What is the main language spoken in {city}?"
    response = openai.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
    )
    return response.choices[0].message.content

In [42]:
spoken_language = find_out_the_main_spoken_language_in_city(city)

In [63]:
def translate_text(text: str, target_language: str, model: str = "gpt-5.6-luna") -> str | None:
    system_prompt = f"""You are a helpful assistant that translates text into {target_language}.
    Respond with the translated text only, without any additional text.
    The response should contain 2000 characters at most.
    Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown."""
    user_prompt = f"Translate the following text into {target_language}, in 2000 characters or less:\n\n{text}"
    response = openai.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
    )
    return response.choices[0].message.content[:4096] if response.choices[0].message.content else None

In [64]:
if summarized_page and spoken_language:
    translated_summary = translate_text(summarized_page, spoken_language)

In [ ]:
def talker(text: str, model: str = "tts-1") -> bytes | None:
    response = openai.audio.speech.create(model=model, voice="coral", input=text[:4096], speed=1)
    return response.content

In [ ]:
# from IPython.display import Audio, display

# if translated_summary:
#     audio_bytes = talker(translated_summary)
#     display(Audio(audio_bytes, autoplay=True))


In [ ]:
def fetch_wikipedia_image_url(subject: str) -> str | None:
    normalized_subject = subject.lower().strip().replace(" ", "_").replace("-", "_")
    print(normalized_subject)
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{normalized_subject}"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
    }
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        return None
    image = response.json().get("originalimage")
    return image["source"] if image else None

In [121]:
image_url = fetch_wikipedia_image_url("new-york")

In [122]:
print(image_url)

None
